# Hypothesis
Scratches are harder to learn, so increasing their frequency may help the generator learn their structure better.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get -qq install -y graphviz
!pip install -q pydot

In [ ]:
from tensorflow.keras.utils import plot_model

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
LATENT_DIM = 100
N_CLASSES = 3
IMG_SHAPE = (128, 128, 1)

In [ ]:

os.makedirs("/content/drive/MyDrive/DSCI602/exp2", exist_ok=True)
os.makedirs("/content/drive/MyDrive/DSCI602/exp2/generated_samples", exist_ok=True)

print("exp2 folders ready.")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/DSCI602/prepared_A"

NORMAL_DIR = os.path.join(BASE_DIR, "normal")
SCRATCH_DIR = os.path.join(BASE_DIR, "scratches")
SPOT_DIR = os.path.join(BASE_DIR, "spots")

IMG_SIZE = 128
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
def list_images(folder):
    files = []
    for f in os.listdir(folder):
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            files.append(os.path.join(folder, f))
    return sorted(files)

normal_files = list_images(NORMAL_DIR)
scratch_files = list_images(SCRATCH_DIR)
spot_files = list_images(SPOT_DIR)

print("Normal:", len(normal_files))
print("Scratches:", len(scratch_files))
print("Spots:", len(spot_files))

In [ ]:
def show_samples(file_list, title, n=8, cmap='gray'):
    chosen = random.sample(file_list, min(n, len(file_list)))
    plt.figure(figsize=(16, 2.5))
    for i, path in enumerate(chosen):
        img = Image.open(path)
        plt.subplot(1, len(chosen), i + 1)
        plt.imshow(img, cmap=cmap)
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples(normal_files, "Original Normal Samples")
show_samples(scratch_files, "Original Scratch Samples")
show_samples(spot_files, "Original Spot Samples")

In [ ]:
def show_before_after_resize(path, title):
    img = Image.open(path).convert("L")
    img_resized = img.resize((IMG_SIZE, IMG_SIZE))

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray')
    plt.title(f"{title} - Original\n{img.size}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_resized, cmap='gray')
    plt.title(f"{title} - Resized\n{img_resized.size}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

show_before_after_resize(normal_files[0], "Normal")
show_before_after_resize(scratch_files[0], "Scratch")
show_before_after_resize(spot_files[0], "Spot")

In [ ]:
all_files = normal_files + scratch_files + spot_files
all_labels = (
    [0] * len(normal_files) +
    [1] * len(scratch_files) +
    [2] * len(spot_files)
)

print("Total files:", len(all_files))
print("Total labels:", len(all_labels))
print("Unique labels:", sorted(set(all_labels)))

## Balanced sampling in code:



In [ ]:
from collections import defaultdict

class_to_files = defaultdict(list)
for path, label in zip(all_files, all_labels):
    class_to_files[label].append(path)

# Different target counts per class
target_counts = {
    0: 700,   # normal
    1: 1000,  # scratch (increase exposure)
    2: 700    # spot
}

def oversample_to_target(file_list, target_count):
    result = []
    while len(result) < target_count:
        result.extend(file_list)
    return result[:target_count]

balanced_files = []
balanced_labels = []

for label in [0, 1, 2]:
    files = class_to_files[label]
    target_count = target_counts[label]

    if len(files) >= target_count:
        selected = random.sample(files, target_count)
    else:
        selected = oversample_to_target(files, target_count)

    balanced_files.extend(selected)
    balanced_labels.extend([label] * len(selected))

print("Balanced dataset size:", len(balanced_files))
print("Balanced class counts:")

for label in [0,1,2]:
    print(label, balanced_labels.count(label))

## Define image loader and normalization

In [ ]:
def load_and_preprocess(path, label):
    img = Image.open(path).convert("L")   # force grayscale
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img = np.array(img).astype("float32")

    # scale from [0,255] to [-1,1]
    img = (img / 127.5) - 1.0

    # add channel dimension: (128,128) -> (128,128,1)
    img = np.expand_dims(img, axis=-1)

    return img, label

## Convert to NumPy arrays

In [ ]:
X = []
y = []

for path, label in zip(balanced_files, balanced_labels):
    img, lab = load_and_preprocess(path, label)
    X.append(img)
    y.append(lab)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X min/max:", X.min(), X.max())
print("Unique labels:", np.unique(y, return_counts=True))

## Visualize normalized images

In [ ]:
label_names = {0: "Normal", 1: "Scratch", 2: "Spot"}

def show_normalized_examples(X, y, n_per_class=5):
    plt.figure(figsize=(12, 6))
    plot_idx = 1
    for row, label in enumerate([0, 1, 2]):
        idxs = np.where(y == label)[0][:n_per_class]
        for col, idx in enumerate(idxs):
            plt.subplot(3, n_per_class, plot_idx)
            plt.imshow((X[idx].squeeze() + 1) / 2.0, cmap='gray')
            if col == 0:
                plt.ylabel(label_names[label], fontsize=12)
            plt.axis("off")
            plot_idx += 1
    plt.tight_layout()
    plt.show()

show_normalized_examples(X, y, n_per_class=5)

# Build Generator

In [ ]:
def build_generator(latent_dim=100, n_classes=3):
    # Label input
    label_input = layers.Input(shape=(1,), name="Generator-Label-Input")
    label_embedding = layers.Embedding(n_classes, 50, name="Generator-Label-Embedding")(label_input)
    label_dense = layers.Dense(4 * 4 * 1, name="Generator-Label-Dense")(label_embedding)
    label_reshape = layers.Reshape((4, 4, 1), name="Generator-Label-Reshape")(label_dense)

    # Latent input
    latent_input = layers.Input(shape=(latent_dim,), name="Generator-Latent-Input")
    latent_dense = layers.Dense(4 * 4 * 256, name="Generator-Latent-Dense")(latent_input)
    latent_act = layers.LeakyReLU(0.2)(latent_dense)
    latent_reshape = layers.Reshape((4, 4, 256), name="Generator-Latent-Reshape")(latent_act)

    # Merge label + latent features
    merge = layers.Concatenate(name="Generator-Combine")([latent_reshape, label_reshape])  # (4,4,257)

    x = layers.Conv2DTranspose(256, kernel_size=4, strides=2, padding="same", name="G_Deconv_8")(merge)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", name="G_Deconv_16")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", name="G_Deconv_32")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", name="G_Deconv_64")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(32, kernel_size=4, strides=2, padding="same", name="G_Deconv_128")(x)
    x = layers.LeakyReLU(0.2)(x)

    output = layers.Conv2D(1, kernel_size=7, activation="tanh", padding="same", name="Generator-Output")(x)

    model = keras.Model([latent_input, label_input], output, name="Generator")
    return model

generator = build_generator(LATENT_DIM, N_CLASSES)
generator.summary()

# Build Discriminator

In [ ]:
def build_discriminator(in_shape=(128, 128, 1), n_classes=3):
    # Label input
    label_input = layers.Input(shape=(1,), name="Discriminator-Label-Input")
    label_embedding = layers.Embedding(n_classes, 50, name="Discriminator-Label-Embedding")(label_input)
    label_dense = layers.Dense(in_shape[0] * in_shape[1], name="Discriminator-Label-Dense")(label_embedding)
    label_reshape = layers.Reshape((in_shape[0], in_shape[1], 1), name="Discriminator-Label-Reshape")(label_dense)

    # Image input
    image_input = layers.Input(shape=in_shape, name="Discriminator-Image-Input")

    # Merge image + label map
    merge = layers.Concatenate(name="Discriminator-Combine")([image_input, label_reshape])  # (128,128,2)

    x = layers.Conv2D(64, kernel_size=4, strides=2, padding="same", name="D_Conv_64")(merge)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same", name="D_Conv_32")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same", name="D_Conv_16")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, kernel_size=4, strides=2, padding="same", name="D_Conv_8")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, kernel_size=4, strides=2, padding="same", name="D_Conv_4")(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    output = layers.Dense(1, activation="sigmoid", name="Discriminator-Output")(x)

    model = keras.Model([image_input, label_input], output, name="Discriminator")
    return model

discriminator = build_discriminator(IMG_SHAPE, N_CLASSES)
discriminator.summary()

## Shape check:

In [ ]:
noise = np.random.normal(0, 1, (4, LATENT_DIM)).astype(np.float32)
labels = np.array([[0], [1], [2], [1]], dtype=np.int32)

fake_imgs = generator.predict([noise, labels], verbose=0)
print("Fake images shape:", fake_imgs.shape)

disc_out = discriminator.predict([fake_imgs, labels], verbose=0)
print("Discriminator output shape:", disc_out.shape)
print("Discriminator outputs:", disc_out.squeeze())

## Visualize untrained generator output

In [ ]:
def show_generated_samples(generator, latent_dim=100, n_classes=3, n_per_class=5):
    plt.figure(figsize=(12, 6))
    plot_idx = 1

    for label in range(n_classes):
        noise = np.random.normal(0, 1, (n_per_class, latent_dim)).astype(np.float32)
        labels = np.full((n_per_class, 1), label, dtype=np.int32)

        gen_imgs = generator.predict([noise, labels], verbose=0)

        for i in range(n_per_class):
            plt.subplot(n_classes, n_per_class, plot_idx)
            plt.imshow((gen_imgs[i].squeeze() + 1) / 2.0, cmap='gray')
            if i == 0:
                plt.ylabel(["Normal", "Scratch", "Spot"][label], fontsize=12)
            plt.axis("off")
            plot_idx += 1

    plt.tight_layout()
    plt.show()

show_generated_samples(generator, LATENT_DIM, N_CLASSES, n_per_class=5)

## Plot G and D:

In [ ]:
from IPython.display import Image, display
plot_model(
    generator,
    to_file="/content/drive/MyDrive/DSCI602/exp2/generator_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=200
)

print("Saved generator architecture diagram.")

plot_model(
    discriminator,
    to_file="/content/drive/MyDrive/DSCI602/exp2/discriminator_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=200
)

print("Saved discriminator architecture diagram.")


display(Image(filename="/content/drive/MyDrive/DSCI602/exp2/generator_architecture.png"))
display(Image(filename="/content/drive/MyDrive/DSCI602/exp2/discriminator_architecture.png"))

## Compile the Discriminator:

In [ ]:
opt_d = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

discriminator.compile(
    loss="binary_crossentropy",
    optimizer=opt_d,
    metrics=["accuracy"]
)

# Build combined GAN:

In [ ]:
discriminator.trainable = False

noise_input = layers.Input(shape=(LATENT_DIM,), name="GAN-Noise-Input")
label_input = layers.Input(shape=(1,), name="GAN-Label-Input")

generated_img = generator([noise_input, label_input])
gan_output = discriminator([generated_img, label_input])

gan_model = keras.Model([noise_input, label_input], gan_output, name="cDCGAN")

opt_g = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

gan_model.compile(
    loss="binary_crossentropy",
    optimizer=opt_g
)

## Plot the GAN

In [ ]:
plot_model(
    gan_model,
    to_file="/content/drive/MyDrive/DSCI602/exp2/gan_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=200
)

print("Saved GAN architecture diagram.")
display(Image(filename="/content/drive/MyDrive/DSCI602/exp2/gan_architecture.png"))

In [ ]:
generator.summary()
discriminator.summary()
gan_model.summary()

## Re-compile Discriminator correctly

In [ ]:
discriminator.trainable = True
for layer in discriminator.layers:
    layer.trainable = True

opt_d = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

discriminator.compile(
    loss="binary_crossentropy",
    optimizer=opt_d,
    metrics=["accuracy"]
)

print("Discriminator recompiled.")
print("Discriminator trainable weights:", len(discriminator.trainable_weights))

## Rebuild combined GAN correctly:

In [ ]:
discriminator.trainable = False
for layer in discriminator.layers:
    layer.trainable = False

noise_input = layers.Input(shape=(LATENT_DIM,), name="GAN-Noise-Input")
label_input = layers.Input(shape=(1,), name="GAN-Label-Input")

generated_img = generator([noise_input, label_input])
gan_output = discriminator([generated_img, label_input])

gan_model = keras.Model([noise_input, label_input], gan_output, name="cDCGAN")

opt_g = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

gan_model.compile(
    loss="binary_crossentropy",
    optimizer=opt_g
)

print("Combined GAN compiled.")

## Check trainable counts:

In [ ]:
print("Generator trainable weights:", len(generator.trainable_weights))
print("Discriminator trainable weights:", len(discriminator.trainable_weights))
print("GAN model trainable weights:", len(gan_model.trainable_weights))

### Helper functions:

In [ ]:
def real_samples(X, y, n):
    idx = np.random.randint(0, X.shape[0], n)
    images = X[idx]
    labels = y[idx].reshape(-1, 1)
    targets = np.ones((n, 1), dtype=np.float32)
    return [images, labels], targets

def latent_vector(latent_dim, n):
    z = np.random.randn(latent_dim * n)
    z = z.reshape(n, latent_dim).astype(np.float32)
    labels = np.random.randint(0, N_CLASSES, n).reshape(-1, 1).astype(np.int32)
    return z, labels

def fake_samples(generator, latent_dim, n):
    z, labels = latent_vector(latent_dim, n)
    images = generator.predict([z, labels], verbose=0)
    targets = np.zeros((n, 1), dtype=np.float32)
    return [images, labels], targets

## Helper for saving generated grid:

In [ ]:
label_names = {0: "Normal", 1: "Scratch", 2: "Spot"}

def save_generated_grid(generator, epoch, latent_dim=100, n_per_class=5, save_dir="generated_samples"):
    os.makedirs(save_dir, exist_ok=True)

    fig, axes = plt.subplots(3, n_per_class, figsize=(12, 6))

    for row, label in enumerate([0, 1, 2]):
        z = np.random.randn(n_per_class, latent_dim).astype(np.float32)
        labels = np.full((n_per_class, 1), label, dtype=np.int32)

        gen_imgs = generator.predict([z, labels], verbose=0)

        for col in range(n_per_class):
            axes[row, col].imshow((gen_imgs[col].squeeze() + 1) / 2.0, cmap="gray")
            axes[row, col].axis("off")

        axes[row, 0].set_title(label_names[label], fontsize=12, pad=8)

    plt.subplots_adjust(left=0.05, wspace=0.05, hspace=0.18)
    filepath = os.path.join(save_dir, f"epoch_{epoch:03d}.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.show()
    print(f"Saved sample grid to: {filepath}")

# Training Loop!

In [ ]:
def train_cdcgan(generator, discriminator, gan_model, X, y,
                 latent_dim=100, n_epochs=100, batch_size=32,
                 save_dir="generated_samples"):

    bat_per_epo = X.shape[0] // batch_size
    half_batch = batch_size // 2

    d_losses_real, d_losses_fake, g_losses = [], [], []

    for epoch in range(1, n_epochs + 1):
        d_loss_real_epoch = []
        d_loss_fake_epoch = []
        g_loss_epoch = []

        # make D trainable for its own updates
        discriminator.trainable = True
        for layer in discriminator.layers:
            layer.trainable = True

        for batch in range(bat_per_epo):
            [X_real, labels_real], y_real = real_samples(X, y, half_batch)
            d_loss_real, _ = discriminator.train_on_batch([X_real, labels_real], y_real)

            [X_fake, labels_fake], y_fake = fake_samples(generator, latent_dim, half_batch)
            d_loss_fake, _ = discriminator.train_on_batch([X_fake, labels_fake], y_fake)

            # freeze D for generator step
            discriminator.trainable = False
            for layer in discriminator.layers:
                layer.trainable = False

            z_input, labels_input = latent_vector(latent_dim, batch_size)
            y_gan = np.ones((batch_size, 1), dtype=np.float32)

            g_loss = gan_model.train_on_batch([z_input, labels_input], y_gan)

            d_loss_real_epoch.append(d_loss_real)
            d_loss_fake_epoch.append(d_loss_fake)
            g_loss_epoch.append(g_loss)

            # turn D back on for next discriminator update
            discriminator.trainable = True
            for layer in discriminator.layers:
                layer.trainable = True

        d_losses_real.append(np.mean(d_loss_real_epoch))
        d_losses_fake.append(np.mean(d_loss_fake_epoch))
        g_losses.append(np.mean(g_loss_epoch))

        print(f"Epoch {epoch}/{n_epochs} | "
              f"D_real: {d_losses_real[-1]:.4f} | "
              f"D_fake: {d_losses_fake[-1]:.4f} | "
              f"G: {g_losses[-1]:.4f}")

        save_generated_grid(generator, epoch, latent_dim=latent_dim, n_per_class=5, save_dir=save_dir)

        # ---- SAVE CHECKPOINTS ----
        if epoch in [19, 20, 21, 38, 39, 40, 41, 42, 43, 44, 45, 79, 80, 81, 82]:
            generator.save(f"/content/drive/MyDrive/DSCI602/exp2/generator_epoch_{epoch:03d}.keras")
            discriminator.save(f"/content/drive/MyDrive/DSCI602/exp2/discriminator_epoch_{epoch:03d}.keras")
            gan_model.save(f"/content/drive/MyDrive/DSCI602/exp2/gan_epoch_{epoch:03d}.keras")
            print(f"Saved model checkpoints at epoch {epoch}")

    return d_losses_real, d_losses_fake, g_losses


## Exp 2 Training till 82 epochs, and saving model progress:

In [ ]:
d_losses_real_exp2, d_losses_fake_exp2, g_losses_exp2 = train_cdcgan(
    generator,
    discriminator,
    gan_model,
    X, y,
    latent_dim=LATENT_DIM,
    n_epochs=82,
    batch_size=32,
    save_dir="/content/drive/MyDrive/DSCI602/exp2/generated_samples_A"
)

# Plotting losses after the training:

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(d_losses_real_exp2, label="D real loss", marker='o', markersize=2, alpha=0.7)
plt.plot(d_losses_fake_exp2, label="D fake loss", marker='x', markersize=2, alpha=0.7)
plt.plot(g_losses_exp2, label="G loss", marker='s', markersize=2, alpha=0.7)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("cDCGAN Training Losses (Experiment 2)")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_plot_path = "/content/drive/MyDrive/DSCI602/exp2/loss_plot_exp2.png"
plt.savefig(loss_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved loss plot to: {loss_plot_path}")


# Saving raw loss arrays too!

np.save("/content/drive/MyDrive/DSCI602/exp2/d_losses_real_exp2.npy", np.array(d_losses_real_exp2))
np.save("/content/drive/MyDrive/DSCI602/exp2/d_losses_fake_exp2.npy", np.array(d_losses_fake_exp2))
np.save("/content/drive/MyDrive/DSCI602/exp2/g_losses_exp2.npy", np.array(g_losses_exp2))

print("Saved raw loss arrays.")

# INFERENCING:

In [ ]:
# =========================
# INFERENCE / GENERATION
# =========================

from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

# ---- Paths to saved generators ----
GEN_PATH_20 = "/content/drive/MyDrive/DSCI602/generator_epoch20.keras"
GEN_PATH_50 = "/content/drive/MyDrive/DSCI602/generator_epoch50.keras"
GEN_PATH_60 = "/content/drive/MyDrive/DSCI602/generator_epoch60.keras"

# ---- Load models ----
generator_infer_20 = load_model(GEN_PATH_20)
generator_infer_50 = load_model(GEN_PATH_50)
generator_infer_60 = load_model(GEN_PATH_60)

print("Loaded generator from:", GEN_PATH_20)
print("Loaded generator from:", GEN_PATH_50)
print("Loaded generator from:", GEN_PATH_60)

# ---- Label mapping ----
label_names = {
    0: "normal",
    1: "scratch",
    2: "spot"
}

# ---- Generate images for one class ----
def generate_images_for_label(generator, label, n_images=10, latent_dim=100,
                              save_dir="/content/drive/MyDrive/DSCI602/inference_outputs",
                              show_grid=True):
    os.makedirs(save_dir, exist_ok=True)

    z = np.random.randn(n_images, latent_dim).astype(np.float32)
    labels = np.full((n_images, 1), label, dtype=np.int32)

    gen_imgs = generator.predict([z, labels], verbose=0)

    class_dir = os.path.join(save_dir, label_names[label])
    os.makedirs(class_dir, exist_ok=True)

    for i in range(n_images):
        img = ((gen_imgs[i].squeeze() + 1) / 2.0 * 255.0).clip(0, 255).astype(np.uint8)
        Image.fromarray(img, mode="L").save(
            os.path.join(class_dir, f"{label_names[label]}_{i:03d}.png")
        )

    print(f"Saved {n_images} images for label {label} ({label_names[label]}) to:")
    print(class_dir)

    if show_grid:
        cols = min(5, n_images)
        rows = int(np.ceil(n_images / cols))
        plt.figure(figsize=(3 * cols, 3 * rows))
        for i in range(n_images):
            plt.subplot(rows, cols, i + 1)
            plt.imshow((gen_imgs[i].squeeze() + 1) / 2.0, cmap="gray")
            plt.axis("off")
        plt.suptitle(f"Generated {label_names[label]} images", fontsize=14)
        plt.tight_layout()
        plt.show()

# ---- Generate all three classes ----
def generate_all_classes(generator, n_images_per_class=10, latent_dim=100,
                         save_dir="/content/drive/MyDrive/DSCI602/inference_outputs_all"):
    for label in [0, 1, 2]:
        generate_images_for_label(
            generator,
            label=label,
            n_images=n_images_per_class,
            latent_dim=latent_dim,
            save_dir=save_dir,
            show_grid=True
        )

# =========================
# EXAMPLES
# =========================

# Example 1: generate only spots from epoch-20 model
# generate_images_for_label(generator_infer_20, label=2, n_images=10,
#                           save_dir="/content/drive/MyDrive/DSCI602/inference_outputs_epoch20")

# Example 2: generate all classes from epoch-50 model
# generate_all_classes(generator_infer_50, n_images_per_class=10,
#                      save_dir="/content/drive/MyDrive/DSCI602/inference_outputs_epoch50")

# Example 3: generate all classes from epoch-60 model
# generate_all_classes(generator_infer_60, n_images_per_class=10,
#                      save_dir="/content/drive/MyDrive/DSCI602/inference_outputs_epoch60")

### Load saved generator

In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

GEN_PATH_60 = "/content/drive/MyDrive/DSCI602/generator_epoch60.keras"
GEN_PATH_50 = "/content/drive/MyDrive/DSCI602/generator_epoch50.keras"
GEN_PATH_20 = "/content/drive/MyDrive/DSCI602/generator_epoch20.keras"
generator_infer_60 = load_model(GEN_PATH_60)
generator_infer_50 = load_model(GEN_PATH_50)
generator_infer_20 = load_model(GEN_PATH_20)

print("Loaded the 60-epoch model generator from:", GEN_PATH_60)
print("Loaded the 50-epoch model generator from:", GEN_PATH_50)
print("Loaded the 20-epoch model generator from:", GEN_PATH_20)

### Label mapping

In [ ]:
label_names = {
    0: "normal",
    1: "scratch",
    2: "spot"
}

### Generate images for one class:


In [ ]:
def generate_images_for_label(generator, label, n_images=10, latent_dim=100,
                              save_dir="/content/drive/MyDrive/DSCI602/inference_outputs_epoch20model",
                              show_grid=True):
    os.makedirs(save_dir, exist_ok=True)

    z = np.random.randn(n_images, latent_dim).astype(np.float32)
    labels = np.full((n_images, 1), label, dtype=np.int32)

    gen_imgs = generator.predict([z, labels], verbose=0)

    class_dir = os.path.join(save_dir, label_names[label])
    os.makedirs(class_dir, exist_ok=True)

    for i in range(n_images):
        img = ((gen_imgs[i].squeeze() + 1) / 2.0 * 255.0).clip(0, 255).astype(np.uint8)
        Image.fromarray(img, mode="L").save(
            os.path.join(class_dir, f"{label_names[label]}_{i:03d}.png")
        )

    print(f"Saved {n_images} images for label {label} ({label_names[label]}) to:")
    print(class_dir)

    if show_grid:
        cols = min(5, n_images)
        rows = int(np.ceil(n_images / cols))
        plt.figure(figsize=(3 * cols, 3 * rows))
        for i in range(n_images):
            plt.subplot(rows, cols, i + 1)
            plt.imshow((gen_imgs[i].squeeze() + 1) / 2.0, cmap="gray")
            plt.axis("off")
        plt.suptitle(f"Generated {label_names[label]} images", fontsize=14)
        plt.tight_layout()
        plt.show()

### If want to generate all three classes:

In [ ]:
def generate_all_classes(generator, n_images_per_class=10, latent_dim=100,
                         save_dir="/content/drive/MyDrive/DSCI602/inference_outputs_all_epoch20model"):
    for label in [0, 1, 2]:
        generate_images_for_label(
            generator,
            label=label,
            n_images=n_images_per_class,
            latent_dim=latent_dim,
            save_dir=save_dir,
            show_grid=True
        )

In [ ]:
generate_images_for_label(generator_infer_20, label=2, n_images=10)

In [ ]:
generate_all_classes(generator_infer_20, n_images_per_class=10)